# Etapa 1 — Ternium: correlaciones, PCA y regresión CS

**Objetivo:** comprender y predecir el tiempo neto **real** de recocido por carga, sin modificar datos de planta.

Primero se auditan datos y correlaciones; luego PCA exploratorio; finalmente regresión múltiple (geometría y una versión explicativa con condiciones del proceso) frente a una referencia mediana. **Sin recocido simulado todavía.**

## 0. Preparación

Ejecutar desde la raíz del repositorio local tras `pip install -r requirements.txt`. En Colab, primero clonar el repositorio y cambiar el directorio actual. **No subir archivos de planta a GitHub.**

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Image
from src.analisis_cs import leer_base, reconstruir_cargas, preparar_muestra, ejecutar

BASE_DEP = Path('data/raw/Recocido -Tec-dep(1).xlsx')
BASE_ORIGINAL = Path('data/raw/Recocido -Tec.xlsx')
SALIDA = Path('results/etapa1')
assert BASE_DEP.exists(), 'Coloca Excel depurado en data/raw (directorio local ignorado por Git)'

## 1. Unidad estadística y limpieza

Se identifican bobinas por carga y posición. Pueden existir filas duplicadas debido a distintas mediciones del mismo material. La unidad de análisis es **carga**: el tiempo neto real se repite entre bobinas, así que no deben contarse como ensayos independientes. Si no existe la base original, solo podemos asegurar que se conservaron filas CS, NO que toda la carga sea CS.

In [ ]:
dep = leer_base(BASE_DEP)
orig = leer_base(BASE_ORIGINAL) if BASE_ORIGINAL.exists() else None
print('Columnas depuradas:', dep.columns.tolist())
cargas, aud = reconstruir_cargas(dep, orig)
print('Auditoría de reconstrucción:')
display(pd.Series(aud, dtype='object').to_frame('valor'))
display(cargas.describe())

## 2. Depuración para modelado

Se filtran valores faltantes, temperatura de saturación no positiva y tiempos de saturación negativos. **No** se excluyen automáticamente cargas con tiempos de recocido grandes. La temperatura permanece en °F y los tiempos en minutos.

In [ ]:
muestra = preparar_muestra(cargas, aud)
print('Cargas válidas:', len(muestra))
display(muestra[['tiempo_real_min','peso_carga_kg','n_rollos','espesor_medio_mm','ancho_medio_mm','temp_sat_f','tiempo_sat_min']].describe())

## 3. Correlación y PCA

La matriz de Pearson revela asociaciones lineales, no causalidad. El PCA utiliza **solo entradas**, estandarizadas, y no incluye tiempo real, tiempo estándar ni eficiencia. Es una técnica no supervisada; no equivale a optimizar.

## 4. Regresión múltiple

Tres referencias: mediana del entrenamiento; regresión geométrica; y regresión con geometría + condiciones térmicas retrospectivas. El tiempo de saturación forma parte del proceso y **no puede interpretarse como predicción prospectiva causal**. La comparación exige R², MAE y RMSE sobre cargas de prueba no vistas.

In [ ]:
auditoria, metricas = ejecutar(BASE_DEP, BASE_ORIGINAL if BASE_ORIGINAL.exists() else None, SALIDA, semilla=42)
display(metricas)
display(pd.read_csv(SALIDA/'coeficientes.csv'))

## 5. Gráficas del análisis

Las gráficas y tablas se exportan únicamente a `results/etapa1/`, que está excluido de Git.

In [ ]:
for nombre in ['01_correlaciones.png','02_pca.png','03_real_vs_predicho.png']:
    print(nombre)
    display(Image(filename=str(SALIDA/nombre)))

## 6. Auditoría del equipo y avance

1. Otro integrante verifica manualmente una muestra de cargas contra Excel y revisa duplicados/valores inválidos.
2. Reproduce métricas R²/MAE sin reutilizar los mismos cálculos de visualización.
3. Comprueba que el modelo no use tiempo estándar, eficiencia ni identificadores como variables explicativas.
4. Compara R² de prueba, CV y errores con la referencia; considera incertidumbre y atípicos.
5. Documenta hallazgos en `docs/auditoria_etapa1.md` antes de construir el optimizador.

**Regla:** no modificar variables hasta obtener el R² más alto del conjunto de prueba; la selección pertenece a CV en entrenamiento y debe respetar plausibilidad física.